In [ ]:
!pip install openai pillow pytesseract

# For Mac
brew install tesseract


In [ ]:
from PIL import Image
import pytesseract

# Load image
image_path = "amp_4.jpg"
img = Image.open(image_path)

# Extract text using pytesseract
extracted_text = pytesseract.image_to_string(img)
print(extracted_text)


In [ ]:
system_prompt = """
You are a medical data assistant. You will be given extracted text from a clinical decision tree (e.g., NCCN guidelines). 
Your task is to output a clean, structured JSON representation of the decision logic. Maintain the decision flow and preserve the clinical meaning.
"""

user_prompt = f"""
Here is the extracted text from a clinical flowchart:

{extracted_text}

Please convert this into a structured JSON format that preserves the logic of the tree. Do not include explanations, only output JSON.
"""

response = openai.ChatCompletion.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

generated_json = response.choices[0].message['content']
print(generated_json)


In [ ]:
with open("tree_output.json", "w") as f:
    f.write(generated_json)


In [ ]:
import os

image_dir = "nccn_images/"
for filename in os.listdir(image_dir):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        img_path = os.path.join(image_dir, filename)
        img = Image.open(img_path)
        text = pytesseract.image_to_string(img)
        
        # Use GPT to convert to JSON
        user_prompt = f"Extracted clinical decision logic:\n\n{text}\n\nConvert to JSON."
        response = openai.ChatCompletion.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0,
        )
        
        json_content = response.choices[0].message['content']
        out_path = os.path.join("json_outputs", filename.replace(".jpg", ".json"))
        with open(out_path, "w") as f:
            f.write(json_content)
